In [1]:
import numpy as np
import CI
import CI_physicist

from electron_integrals import *

In [2]:
# Number of orbitals (without spin)
num_orbitals = 5
# Number of electrons
num_electrons = 3
#Include spin?
include_spin = False

num_spin_orbitals = (1+int(include_spin))*num_orbitals

Calculate electron integrals

In [3]:
x_max = 10
num_points = 1000

x = np.linspace(-x_max,x_max,num_points)

pot = GaussianWell(w=100, a=1, center=0)
#pot = HOPotential()

spf, h = get_spf_and_diag_h(num_orbitals, x, pot)

if include_spin:
    #Add spin
    h = np.kron(h, np.eye(2,2))

g = coulomb_interaction_matrix_elements(spf, spf, x, x, kappa = 1, a=0.01)

if include_spin:
    #Add spin
    g = np.kron(g, np.einsum("pr,qs->pqrs",np.eye(2,2), np.eye(2,2)))

g_chemist = g.transpose(0,2,1,3)

In [4]:
%%timeit
H_chemist = CI.AddressHamiltonian(num_spin_orbitals, num_electrons, h, g_chemist).get_hamiltonian()
E_chemist, _ = np.linalg.eigh(H_chemist)
#print(E_chemist)

2 s ± 24.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
%%timeit
H_physicist = CI_physicist.AddressHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_physicist, _ = np.linalg.eigh(H_physicist)
#print(E_physicist)

1.99 s ± 24 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
%%timeit
H_SC_physicist = CI_physicist.SlaterCondonHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_SC_physicist, C = np.linalg.eigh(H_SC_physicist)
#print(E_SC_physicist)

305 ms ± 14.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [4]:
H_chemist = CI.AddressHamiltonian(num_spin_orbitals, num_electrons, h, g_chemist).get_hamiltonian()
E_chemist, _ = np.linalg.eigh(H_chemist)
H_physicist = CI_physicist.AddressHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_physicist, _ = np.linalg.eigh(H_physicist)
H_SC_physicist = CI_physicist.SlaterCondonHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_SC_physicist, C = np.linalg.eigh(H_SC_physicist)

In [5]:
np.testing.assert_allclose(E_chemist, E_physicist)
np.testing.assert_allclose(E_chemist, E_SC_physicist)

In [40]:
import DirectCI
import importlib
importlib.reload(DirectCI)
from DirectCI import DirectCI

In [41]:
dCI = DirectCI(h, g, num_orbitals, num_electrons, num_beta_electrons=0)


In [20]:
rng = np.random.default_rng()
C_test = rng.random(len(E_SC_physicist)).astype(np.cdouble)
C_test = np.divide(C_test, np.sqrt(C_test.T.conj()@C_test))
C_test=C_test.reshape(-1,1)

np.testing.assert_allclose(dCI.get_sigma(C_test), H_SC_physicist@C_test)

In [36]:
DirectCI(h, g, 10, 5, 1)
